In [1]:
import pandas as pd
import json
import warnings
import re

warnings.filterwarnings("ignore")

try:
    df = pd.read_csv("hinglish_textosql_4500.csv")
    print(f"Success! Loaded raw dataset with {len(df)} base rows.")
    print(f"Expected total variant variations: {len(df) * 3}")
except FileNotFoundError:
    print("Error: 'hinglish_textosql_4500.csv' not found. Please upload it to your Colab session files.")

Success! Loaded raw dataset with 1469 base rows.
Expected total variant variations: 4407


In [3]:
import pandas as pd
import json
import warnings
import re
from google.colab import files

warnings.filterwarnings("ignore")

try:
    df = pd.read_csv("hinglish_textosql_4500.csv")
    print(f"Loaded raw dataset with {len(df)} base rows.")
except FileNotFoundError:
    print("Error: 'hinglish_textosql_4500.csv' not found.")

def clean_hinglish_pipeline(dataframe):
    initial_count = len(dataframe)
    dataframe = dataframe.dropna(subset=['english', 'hinglish_light', 'hinglish_natural', 'hinglish_hindi_heavy', 'sql'])
    columns_to_clean = ['english', 'hinglish_light', 'hinglish_natural', 'hinglish_hindi_heavy', 'sql']

    for col in columns_to_clean:
        dataframe[col] = dataframe[col].astype(str).str.strip()
        dataframe[col] = dataframe[col].str.replace(r'^["\']|["\']$', '', regex=True)
        dataframe[col] = dataframe[col].str.replace('|||', '', regex=False)

    invalid_mask = (
        dataframe['hinglish_light'].str.contains("Light:", case=False, na=False) |
        dataframe['hinglish_natural'].str.contains("Natural:", case=False, na=False) |
        dataframe['hinglish_hindi_heavy'].str.contains("Heavy:", case=False, na=False)
    )

    cleaned_df = dataframe[~invalid_mask]
    final_count = len(cleaned_df)

    print(f"Initial Rows: {initial_count}")
    print(f"Anomalous Rows Dropped: {initial_count - final_count}")
    print(f"Pristine Base Rows Retained: {final_count}")
    print(f"Total Cleaned Evaluation Queries: {final_count * 3}")

    return cleaned_df.reset_index(drop=True)

cleaned_df = clean_hinglish_pipeline(df)

if not cleaned_df.empty:
    sample = cleaned_df.sample(1).iloc[0]
    print(f"English Base:\n{sample['english']}\n")
    print(f"Hinglish Light:\n{sample['hinglish_light']}\n")
    print(f"Hinglish Natural:\n{sample['hinglish_natural']}\n")
    print(f"Hinglish Hindi-Heavy:\n{sample['hinglish_hindi_heavy']}\n")
    print(f"Target SQL:\n{sample['sql']}")

cleaned_df.to_csv("hinglish_textosql_cleaned.csv", index=False, encoding="utf-8")

cleaned_json_list = cleaned_df.to_dict(orient="records")
with open("hinglish_textosql_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_json_list, f, indent=2, ensure_ascii=False)

files.download("hinglish_textosql_cleaned.csv")
files.download("hinglish_textosql_cleaned.json")

Loaded raw dataset with 1469 base rows.
Initial Rows: 1469
Anomalous Rows Dropped: 1
Pristine Base Rows Retained: 1468
Total Cleaned Evaluation Queries: 4404
English Base:
What are the name and description for location code x?

Hinglish Light:
What are name aur prakar for location code x?

Hinglish Natural:
Location ka name aur description kya hain code x ke liye?

Hinglish Hindi-Heavy:
Kya location code x ke liye name aur prakar zarur hai?

Target SQL:
SELECT location_name ,  location_description FROM Ref_locations WHERE location_code  =  "x


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>